In [9]:
import json
import requests
from pathlib import Path
from tqdm import tqdm
from bs4 import BeautifulSoup

# ---------------------------
# Scraping function
# ---------------------------
def scrape_text_from_url(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside', 'form']):
            tag.decompose()

        text_blocks = []
        for tag in soup.find_all(['p', 'li']):
            text = tag.get_text(strip=True)
            if text:
                text_blocks.append(text)

        return text_blocks

    except Exception as e:
        print(f"[ERROR] Failed to scrape {url}: {e}")
        return []

# ---------------------------
# Paths
# ---------------------------
input_path = "/Users/ojas/Desktop/Research/LCS2/Factuality Bias Reasoning Traces Arghodeep/Datsets/Averitec.json"
output_path = "/Users/ojas/Desktop/Research/LCS2/Factuality Bias Reasoning Traces Arghodeep/Averitec_extracted_withscrape.json"

# ---------------------------
# Load JSON data
# ---------------------------
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

output_data = []

# ---------------------------
# Process each entry
# ---------------------------
for entry in tqdm(data):
    claim = entry.get("claim", "").strip()
    label = entry.get("label", "").strip()
    justification = entry.get("justification", "").strip()
    urls = set()
    for q in entry.get("questions", []):
        for ans in q.get("answers", []):
            url = ans.get("source_url")
            if url:
                urls.add(url)

    evidence_texts = []
    for qa in entry.get("questions", []):
        question = qa.get("question", "").strip()
        answers = qa.get("answers", [])
        for answer_entry in answers:
            answer = answer_entry.get("answer", "").strip()
            if question and answer:
                evidence_texts.append(f"{question} ; {answer}")

    if justification:
        evidence_texts.append(justification)

    # 🔍 Scrape from sources if available
    scraped_info = []
    for source in urls:
        url = source.strip()
        if url:
            scraped_texts = scrape_text_from_url(url)
            if scraped_texts:
                scraped_info.extend(scraped_texts)

    # Final result object
    result = {
        "claim": claim,
        "label": label,
        "evidence_text": evidence_texts,
        "scraped_info": scraped_info  # ✅ New field added
    }
    output_data.append(result)

# ---------------------------
# Save updated data
# ---------------------------
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"✅ Extracted {len(output_data)} entries to: {output_path}")

  0%|          | 8/3068 [00:23<3:01:33,  3.56s/it]

[ERROR] Failed to scrape https://web.archive.org/web/20201209174955/https://www.congress.gov/bill/107th-congress/house-joint-resolution/114/actions: HTTPSConnectionPool(host='web.archive.org', port=443): Read timed out. (read timeout=10)


  0%|          | 12/3068 [00:50<4:24:20,  5.19s/it]

[ERROR] Failed to scrape https://www.reuters.com/article/us-usa-economy/u-s-economic-growth-in-2018-misses-trumps-3-percent-target-idUSKCN1QH0HO: 401 Client Error: HTTP Forbidden for url: https://www.reuters.com/article/us-usa-economy/u-s-economic-growth-in-2018-misses-trumps-3-percent-target-idUSKCN1QH0HO
[ERROR] Failed to scrape https://www.imf.org/en/News/Articles/2016/10/03/AM2016-NA100416-WEO: HTTPSConnectionPool(host='www.imf.org', port=443): Read timed out. (read timeout=10)
[ERROR] Failed to scrape https://nrf.com/research/monthly-economic-review-september-2019: 404 Client Error: Not Found for url: https://nrf.com/research/monthly-economic-review-september-2019


  0%|          | 13/3068 [01:06<7:12:18,  8.49s/it]

...

 92%|█████████▏| 2835/3068 [2:00:57<00:01, 200.89it/s]

[ERROR] Failed to scrape https://pmg.org.za/briefing/18970/: HTTPSConnectionPool(host='pmg.org.za', port=443): Max retries exceeded with url: /briefing/18970/ (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x11c9cc7d0>: Failed to resolve 'pmg.org.za' ([Errno 8] nodename nor servname provided, or not known)"))
[ERROR] Failed to scrape https://www.afrobarometer.org/wp-content/uploads/2022/02/ab_r7_dispatchno254_land_redistribution_in_south_africa.pdf: HTTPSConnectionPool(host='www.afrobarometer.org', port=443): Max retries exceeded with url: /wp-content/uploads/2022/02/ab_r7_dispatchno254_land_redistribution_in_south_africa.pdf (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x11c9cdbd0>: Failed to resolve 'www.afrobarometer.org' ([Errno 8] nodename nor servname provided, or not known)"))
[ERROR] Failed to scrape https://worldpoverty.io/: HTTPSConnectionPool(host='worldpoverty.io', port=443): Max retries exceeded with url: / (C

100%|██████████| 3068/3068 [2:00:57<00:00,  2.37s/it] 

[ERROR] Failed to scrape https://www.theaa.com/driving-advice/legal/penalty-charge-notice: HTTPSConnectionPool(host='www.theaa.com', port=443): Max retries exceeded with url: /driving-advice/legal/penalty-charge-notice (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x13f583750>: Failed to resolve 'www.theaa.com' ([Errno 8] nodename nor servname provided, or not known)"))
[ERROR] Failed to scrape https://twitter.com/MartinSLewis/status/735218085012115456: HTTPSConnectionPool(host='twitter.com', port=443): Max retries exceeded with url: /MartinSLewis/status/735218085012115456 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x13f583250>: Failed to resolve 'twitter.com' ([Errno 8] nodename nor servname provided, or not known)"))
[ERROR] Failed to scrape https://web.archive.org/web/20210721095645/https://www.moneysavingexpert.com/reclaim/private-parking-tickets/: HTTPSConnectionPool(host='web.archive.org', port=443): Max retries e

✅ Extracted 3068 entries to: /Users/ojas/Desktop/Research/LCS2/Factuality Bias Reasoning Traces Arghodeep/Averitec_extracted_withscrape.json


In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse

def is_visible_text(tag):
    # Filters out unwanted tags
    return tag.name in ['p', 'li'] and tag.get_text(strip=True)

def scrape_text_from_url(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove scripts, styles, navs, footers, headers, ads
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside', 'form']):
            tag.decompose()

        # Collect all visible <p> and <li> text
        text_blocks = []
        for tag in soup.find_all(['p', 'li']):
            text = tag.get_text(strip=True)
            if text:
                text_blocks.append(text)

        return text_blocks

    except Exception as e:
        print(f"[ERROR] Failed to scrape {url}: {e}")
        return []

# Example usage
urls = [
    "https://www.history.com/topics/germany/eugenics",
    "https://sites.utexas.edu/ransomcentermagazine/2012/04/20/tmwi/",
    "https://en.wikipedia.org/wiki/Jhumpa_Lahiri"
]

all_results = {}

for url in urls:
    print(f"Scraping: {url}")
    texts = scrape_text_from_url(url)
    all_results[url] = texts
    print(f"Extracted {len(texts)} blocks\n")

all_results

Scraping: https://www.history.com/topics/germany/eugenics
Extracted 93 blocks

Scraping: https://sites.utexas.edu/ransomcentermagazine/2012/04/20/tmwi/
Extracted 10 blocks

Scraping: https://en.wikipedia.org/wiki/Jhumpa_Lahiri
Extracted 365 blocks



{'https://www.history.com/topics/germany/eugenics': ['By:HISTORY.com Editors',
  'HISTORY.com Editors',
  'Sir Francis Galton, founder of the science of eugenics. (Bettmann / Getty Images)',
  'Published:November 15, 2017',
  'Last Updated:May 28, 2025',
  'Print',
  'Share',
  'Table of contents',
  '1',
  'Francis Galton',
  '2',
  'Eugenics in America',
  '3',
  'Forced Sterilizations',
  '4',
  'Adolf Hitler and Eugenics',
  '5',
  'Josef Mengele',
  '6',
  'Genetic Engineering',
  '7',
  'Sources',
  'Table of contents',
  'Eugenics is the practice or advocacy of improving the human species by selectively mating people with specific desirable hereditary traits. It aims to reduce human suffering by “breeding out” disease, disabilities and so-called undesirable characteristics from the human population. Early supporters of eugenics believed people inherited mental illness, criminal tendencies and even poverty, and that these conditions could be bred out of the gene pool.',
  'Histor